# Intermediate Pairwise Correlation Experiments

This notebook is for exploratory pairwise correlations: all molecule pairs, pairs with small heavy-atom-count difference, size-stratified checks, and diagnostic plots.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from molecular_similarity_experiment import (
    collect_valid_3d_molecules,
    compute_pairwise_metrics,
    compute_voxels_for_molecules,
    compute_zernike_for_all,
    coverage_pretty,
    filter_by_heavy_atoms,
    load_zernike_cache,
    pairwise_spearman_summary,
)

CSV_FILE = "chembl_raw_dump.csv"
SEED = 10
GROUP = (15, 20)
TARGET_N = 500
GRID_WIDTH = 0.2
MAX_ORDER = 6
PARAM = {"default_radius_multiplier": 1.6}

## Build One Seed/Group Dataset

Run this once before the pairwise sections below. It builds the same 3D molecules and Zernike descriptors as the coverage experiment.

In [ ]:
df = pd.read_csv(CSV_FILE)
min_atoms, max_atoms = GROUP
cache = load_zernike_cache(MAX_ORDER)

df_group = filter_by_heavy_atoms(df, min_atoms, max_atoms)
molecules, chembl_ids, smiles_list = collect_valid_3d_molecules(df_group, target_n=TARGET_N, seed=SEED)
voxelcubes, corners, *_ = compute_voxels_for_molecules(molecules, cube_size=64, radius_scale=0.8)
zernike_descriptors = compute_zernike_for_all(voxelcubes, corners, GRID_WIDTH, cache, PARAM)

len(molecules), len(zernike_descriptors)

## Correlation Over All Pairs

In [ ]:
df_pairs_all = compute_pairwise_metrics(molecules, zernike_descriptors, max_atom_diff=None)
df_pairs_all.to_csv(f"pairwise_pairs_seed{SEED}_{min_atoms}_{max_atoms}_all.csv", index=False)

pairwise_spearman_summary(df_pairs_all)

## Correlation for Pairs with Small Atom Difference

In [ ]:
MAX_ATOM_DIFF = 5
df_pairs_atomdiff = df_pairs_all[df_pairs_all["atom_diff"] <= MAX_ATOM_DIFF].copy()
df_pairs_atomdiff.to_csv(
    f"pairwise_pairs_seed{SEED}_{min_atoms}_{max_atoms}_atomdiff{MAX_ATOM_DIFF}.csv",
    index=False,
)

pairwise_spearman_summary(df_pairs_atomdiff)

## Size-Stratified Correlations

In [ ]:
threshold = df_pairs_atomdiff["pair_size"].median()
small_pairs = df_pairs_atomdiff[df_pairs_atomdiff["pair_size"] <= threshold]
large_pairs = df_pairs_atomdiff[df_pairs_atomdiff["pair_size"] > threshold]

summary_by_size = pd.DataFrame([
    {"subset": "small", **pairwise_spearman_summary(small_pairs).set_index("metric")["spearman"].to_dict()},
    {"subset": "large", **pairwise_spearman_summary(large_pairs).set_index("metric")["spearman"].to_dict()},
])
summary_by_size

## Diagnostic Plots

In [ ]:
def plot_vs_tanimoto(df_pairs, metric, title):
    corr = df_pairs[metric].corr(df_pairs["tanimoto"], method="spearman")
    plt.figure(figsize=(7, 5))
    plt.scatter(df_pairs[metric], df_pairs["tanimoto"], alpha=0.4)
    plt.xlabel(metric)
    plt.ylabel("tanimoto")
    plt.title(f"{title}\nSpearman = {corr:.3f}")
    plt.grid(True)
    plt.show()

plot_vs_tanimoto(df_pairs_atomdiff, "zernike", f"Zernike vs Tanimoto, atom diff <= {MAX_ATOM_DIFF}")
plot_vs_tanimoto(df_pairs_atomdiff, "rdkit", f"RDKit vs Tanimoto, atom diff <= {MAX_ATOM_DIFF}")
plot_vs_tanimoto(df_pairs_atomdiff, "combined", f"Combined vs Tanimoto, atom diff <= {MAX_ATOM_DIFF}")

## Multi-Seed Pairwise Correlation Template

This helper keeps exploratory correlation runs separate from the final top-k coverage pipeline.

In [ ]:
def run_pairwise_correlation_for_seed_group(seed, group, max_atom_diff=None, target_n=TARGET_N):
    min_atoms, max_atoms = group
    df_group = filter_by_heavy_atoms(pd.read_csv(CSV_FILE), min_atoms, max_atoms)
    molecules, _, _ = collect_valid_3d_molecules(df_group, target_n=target_n, seed=seed)
    voxelcubes, corners, *_ = compute_voxels_for_molecules(molecules, cube_size=64, radius_scale=0.8)
    descriptors = compute_zernike_for_all(voxelcubes, corners, GRID_WIDTH, load_zernike_cache(MAX_ORDER), PARAM)
    pairs = compute_pairwise_metrics(molecules, descriptors, max_atom_diff=max_atom_diff)
    summary = pairwise_spearman_summary(pairs)
    summary["seed"] = seed
    summary["group"] = f"{min_atoms}-{max_atoms}"
    summary["max_atom_diff"] = max_atom_diff
    summary["n_pairs"] = len(pairs)
    return summary

# Example:
# summaries = [run_pairwise_correlation_for_seed_group(seed, (15, 20), max_atom_diff=5) for seed in [10, 20, 30]]
# pd.concat(summaries, ignore_index=True)